In [ ]:
!pip install -q -U transformers scikit-learn

import os
import json
import shutil
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive

drive.mount('/content/drive')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị huấn luyện:", device)


DATA_DIR = "/content/drive/MyDrive/data_helpdesk"
if not os.path.exists(f"{DATA_DIR}/train_final.csv"):
    DATA_DIR = "/content/drive/MyDrive/NLP_Helpdesk"

SAVE_DIR = f"{DATA_DIR}/saved_model"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Thư mục dữ liệu:", DATA_DIR)

MODEL_NAME = "vinai/phobert-large"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 8
LR = 1.5e-5
PATIENCE = 3

train_df = pd.read_csv(f"{DATA_DIR}/train_final.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val_final.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test_final.csv")


CORE_QUEUES = {
    "Technical Support",
    "Product Support",
    "Customer Service",
    "IT Support",
    "Billing and Payments",
    "Returns and Exchanges",
    "Service Outages and Maintenance",
    "Sales and Pre-Sales",
    "Human Resources",
}

def map_queue(q):
    return q if q in CORE_QUEUES else "Other Support"

for df in [train_df, val_df, test_df]:
    df["queue_grouped"] = df["queue"].apply(map_queue)

PRIORITY_LABELS = ["very_low", "low", "medium", "high", "critical"]
TYPE_LABELS     = sorted(train_df["type"].dropna().unique().tolist())
QUEUE_LABELS    = sorted(train_df["queue_grouped"].unique().tolist())

prio2id  = {l: i for i, l in enumerate(PRIORITY_LABELS)}
type2id  = {l: i for i, l in enumerate(TYPE_LABELS)}
queue2id = {l: i for i, l in enumerate(QUEUE_LABELS)}

print(f"Số lớp sau tối ưu -> Priority: {len(PRIORITY_LABELS)} | Type: {len(TYPE_LABELS)} | Queue: {len(QUEUE_LABELS)}")

label_encoders = {
    "priority_labels": PRIORITY_LABELS,
    "type_labels": TYPE_LABELS,
    "queue_labels": QUEUE_LABELS,
    "max_len": MAX_LEN,
    "model_name": MODEL_NAME,
    "arch_version": "v2_proj_hybrid_pool",
}
with open(f"{SAVE_DIR}/label_encoders.json", "w", encoding="utf-8") as f:
    json.dump(label_encoders, f, ensure_ascii=False, indent=2)

# Dataset & DataLoader
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HelpdeskDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text_phobert"].fillna("").tolist()
        self.prios = [prio2id[x] for x in df["priority"]]
        self.queues = [queue2id[x] for x in df["queue_grouped"]]
        self.types = [type2id[x] if pd.notna(x) and x in type2id else -100 for x in df["type"]]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "priority": torch.tensor(self.prios[idx], dtype=torch.long),
            "queue": torch.tensor(self.queues[idx], dtype=torch.long),
            "type": torch.tensor(self.types[idx], dtype=torch.long),
        }

train_loader = DataLoader(HelpdeskDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(HelpdeskDataset(val_df), batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader  = DataLoader(HelpdeskDataset(test_df), batch_size=BATCH_SIZE * 2, shuffle=False)

# Kiến trúc Multi-Task PhoBERT V2 (Hybrid CLS+Mean Pooling + Projection Heads)
class MultiTaskPhoBERT(nn.Module):
    def __init__(self, model_name, num_types, num_queues, num_prios):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.3)

        self.proj_type = nn.Sequential(
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.2)
        )
        self.proj_queue = nn.Sequential(
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.2)
        )
        self.proj_prio = nn.Sequential(
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.2)
        )

        self.head_type  = nn.Linear(256, num_types)
        self.head_queue = nn.Linear(256, num_queues)
        self.head_prio  = nn.Linear(256, num_prios)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        cls_emb = last_hidden[:, 0, :]
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        mean_emb = torch.sum(last_hidden * mask_expanded, 1) / torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled = self.dropout(0.5 * cls_emb + 0.5 * mean_emb)

        return (
            self.head_type(self.proj_type(pooled)),
            self.head_queue(self.proj_queue(pooled)),
            self.head_prio(self.proj_prio(pooled)),
        )

# Focal Loss + Ordinal MSE Loss cho Priority
class FocalOrdinalPriorityLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, ordinal_weight=0.15):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ordinal_weight = ordinal_weight

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-F.cross_entropy(logits, targets, reduction="none"))
        focal_loss = (((1.0 - pt) ** self.gamma) * ce_loss).mean()

        probs = torch.softmax(logits, dim=1)
        levels = torch.arange(logits.size(1), device=logits.device, dtype=torch.float)
        expected_level = torch.sum(probs * levels, dim=1)
        ordinal_loss = F.mse_loss(expected_level, targets.float())

        return focal_loss + self.ordinal_weight * ordinal_loss

model = MultiTaskPhoBERT(
    MODEL_NAME, len(TYPE_LABELS), len(QUEUE_LABELS), len(PRIORITY_LABELS)
).to(device)

prio_alpha = torch.tensor([2.0, 1.35, 0.75, 0.85, 2.2], dtype=torch.float).to(device)
criterion_prio  = FocalOrdinalPriorityLoss(alpha=prio_alpha, gamma=1.5, ordinal_weight=0.15)
criterion_queue = nn.CrossEntropyLoss(label_smoothing=0.05)
criterion_type  = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.05)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

# Hàm đánh giá
def evaluate(loader):
    model.eval()
    p_true, p_pred = [], []
    q_true, q_pred = [], []
    t_true, t_pred = [], []

    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                logits_t, logits_q, logits_p = model(ids, mask)

            p_true.extend(batch["priority"].numpy())
            p_pred.extend(logits_p.argmax(dim=1).cpu().numpy())

            q_true.extend(batch["queue"].numpy())
            q_pred.extend(logits_q.argmax(dim=1).cpu().numpy())

            mask_t = batch["type"] != -100
            if mask_t.any():
                t_true.extend(batch["type"][mask_t].numpy())
                t_pred.extend(logits_t.argmax(dim=1).cpu()[mask_t].numpy())

    metrics = {
        "prio_acc":  accuracy_score(p_true, p_pred),
        "prio_f1":   f1_score(p_true, p_pred, average="macro"),
        "queue_acc": accuracy_score(q_true, q_pred),
        "queue_f1":  f1_score(q_true, q_pred, average="macro"),
        "type_acc":  accuracy_score(t_true, t_pred),
        "type_f1":   f1_score(t_true, t_pred, average="macro"),
    }
    return metrics, (p_true, p_pred), (q_true, q_pred), (t_true, t_pred)

# Vòng lặp huấn luyện kèm Early Stopping
best_score = 0.0
no_improve = 0
print(f"\nBắt đầu huấn luyện {MODEL_NAME} (V2)...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y_t = batch["type"].to(device)
        y_q = batch["queue"].to(device)
        y_p = batch["priority"].to(device)

        with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
            logits_t, logits_q, logits_p = model(ids, mask)
            loss = (
                criterion_prio(logits_p, y_p)
                + 0.35 * criterion_queue(logits_q, y_q)
                + 0.60 * criterion_type(logits_t, y_t)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    val_m, _, _, _ = evaluate(val_loader)
    combined_f1 = 0.55 * val_m["prio_f1"] + 0.25 * val_m["type_f1"] + 0.20 * val_m["queue_f1"]

    print(
        f"Epoch {epoch}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | "
        f"Val Prio Acc: {val_m['prio_acc']:.4f} (F1: {val_m['prio_f1']:.4f}) | "
        f"Queue F1: {val_m['queue_f1']:.4f} | Type F1: {val_m['type_f1']:.4f}"
    )

    if combined_f1 > best_score:
        best_score = combined_f1
        no_improve = 0
        torch.save(model.state_dict(), f"{SAVE_DIR}/model_weights.pt")
        tokenizer.save_pretrained(SAVE_DIR)
        print(f" Đã lưu mô hình tốt nhất (Combined F1 = {best_score:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  ⏹Dừng sớm (Early Stopping) tại Epoch {epoch} do không cải thiện thêm.")
            break


print("\n Kết quả đánh giá trên tập Test (V2):")
model.load_state_dict(torch.load(f"{SAVE_DIR}/model_weights.pt"))
test_m, (pt, pp), (qt, qp), (tt, tp) = evaluate(test_loader)
print(f"- Priority : Accuracy = {test_m['prio_acc']*100:.2f}% | Macro-F1 = {test_m['prio_f1']*100:.2f}%")
print(f"- Queue    : Accuracy = {test_m['queue_acc']*100:.2f}% | Macro-F1 = {test_m['queue_f1']*100:.2f}%")
print(f"- Type     : Accuracy = {test_m['type_acc']*100:.2f}% | Macro-F1 = {test_m['type_f1']*100:.2f}%")
print("\nChi tiết phân loại Priority trên Test:")
print(classification_report(pt, pp, target_names=PRIORITY_LABELS, digits=4))

zip_path = shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)
print(f"\nĐã nén xong gói mô hình V2 tại: {zip_path}")